# Strategy Analysis - Plot simple moving averages
- Database: PostreSQL
- Dataframe: Polars
- Stock: Multiple stocks but will filter on one.
- Time frame: 2 years
- Only uses close

In [1]:
# Perform imports
import polars as pl
from datetime import date, datetime
import altair as alt

from securities_load.securities.polar_table_functions import (retrieve_close_using_currency_tickers_dates)

In [2]:
# Define variables
currency = "USD"
symbols = ["AAPL", "GOOG"]
today = datetime.now()
min_date = date(today.year - 2, 1, 1)
start = min_date.strftime("%Y-%m-%d")
end = today.strftime("%Y-%m-%d")
filter_on = "GOOG"

In [3]:
# Get the close data for all symbols
pl_df = retrieve_close_using_currency_tickers_dates(
    currency, symbols, start, end)
# pl_df.head()

In [4]:
# Define moving averages over each symbol
pl_df = pl_df.with_columns(
    pl.col("close").rolling_mean(window_size=10).over("ticker").alias("close_MA10"),
    pl.col("close").rolling_mean(window_size=50).over("ticker").alias("close_MA50"),
)

In [5]:
# Plot a chart of the close for all symbols
line_chart = (
    alt.Chart(pl_df)
    .mark_line(strokeWidth=1)
    .encode(x=alt.X("date:T",
                    title='',
                    axis=alt.Axis(labelAngle=-45, tickColor="#09080B", domainColor="#09080B", grid=False, labelFontSize=10, titleFontSize=12)),
            y=alt.Y("close:Q",
                    title='Close',
                    scale=alt.Scale(zero=False),
                    axis=alt.Axis(format='$.2f', tickColor="#09080B", domainColor="#09080B", gridColor="#9DCDF8", grid=True, labelFontSize=10, titleFontSize=12)
                    ),
            color="ticker:N",
            tooltip=['date:T', 'close:Q'])
    ).properties(title={'text': 'Stock Price Analysis',
                        'subtitle': ['Comparison of multiple tickers'],
                        'fontSize': 16
                },
                width=500,
                height=300,
    )
line_chart

alt.Chart(...)

In [6]:
# Filter for one stock and unpivot the close and MAs on the close
filtered_df = pl_df.filter(pl.col("ticker") == filter_on)
long_data = filtered_df.unpivot(index='date', on=['close', 'close_MA10', 'close_MA50'], variable_name='type', value_name='value')

In [7]:
# Plot the close and MAs
line_chart = (
    alt.Chart(long_data)
    .mark_line(strokeWidth=1)
    .encode(x=alt.X("date:T",
                    title='',
                    axis=alt.Axis(labelAngle=-45, tickColor="#09080B", domainColor="#09080B", grid=False, labelFontSize=10, titleFontSize=12)),
            y=alt.Y("value:Q",
                    title='Price',
                    scale=alt.Scale(zero=False),
                    axis=alt.Axis(format='$.2f', tickColor="#09080B", domainColor="#09080B", gridColor="#9DCDF8", grid=True, labelFontSize=10, titleFontSize=12)
                    ),
            color="type:N",
            tooltip=['date:T', 'value:Q'])
    ).properties(title={'text': f'Stock Price Analysis for {filter_on}',
                        'subtitle': ['Comparison of multiple tickers'],
                        'fontSize': 16
                },
                width=500,
                height=300,
    )
line_chart

alt.Chart(...)